## Как писать тесты?

[Общие требования к коду](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=12039618894)

Все валидационные тесты должны находиться в директории `val_test` и быть распределены по директориям в зависимости от блока тестов. Например, тест на ключевую метрику должен лежать в директории `llm_val/val_test/model_quality`.

Каждый тест должен возвращать словарь следующего вида: `{'report': dict, 'precomputed': dict}`, где:
- `precomputed` - это словарь с данными, которые валидатор хочет сохранить;
- `report` - словарь следующего вида: `{'semaphore': str, 'result_plots': list, 'result_dataframes': list}`, где:
    - `semaphore` - цвет светофора. Одно из четырех значений: `['green', 'yellow', 'red', 'gray']`;
    - `result_plots` - список с графиками для отчета;
    - `result_dataframes` - список с DataFrame(таблицами) для отчета.
    
Вкратце:
- в `precomputed` мы сохраняем то, что необходимо для получения итогового результата теста (при подстановке значений из `precomputed` в функцию должен получиться корректный результат теста) (см. "Запуск с использованием предподсчитанных значений" в "5.0.Test_example");
- в `report` мы сохраняем то, что хотели бы включить в excel-отчет. 

В каждом тесте, где есть светофор, должен быть параметр `is_info`, чтобы выключать светофор.

В конце параметров каждого теста должен быть `**kwargs`.

### Этапы написания теста

0. **Определяемся с названием теста**. Название будет фигурировать в названии файла и функции. Название должно быть осмысленным, то есть чтобы по названию теста было понятно, что это за тест и что он делает. В названии теста должно быть не более 32 символов.
1. **Создаем файл valtest_*название_теста*.py**. Файл создаем в соответствующей директории в `llm_val/val_test`. Например, если тест на стабильность, то файл будет лежать в `llm_val/val_test/stability`;
2. **Пишем логику теста**. Логика теста должна быть декомпозирована, если это требуется. Общие правила написания кода см. по ссылке выше.
3. **Создаем report**. Создание report'а должно представлять собой отдельную функцию с префиксом report_. Структуру report'a см. выше.
4. **Сохраняем необходимые значения в precomputed**. `precomputed` - это словарь с данными, которые валидатор хочет сохранить. Также он используется в случае, когда необходимо провести тест, но не хочется пересчитывать все значения.
5. **Добавляем значения из precomputed в параметры функции**. Это необходимо для корректной работы precomputed в случае, если нет необходимости пересчитывать значения теста, но тест провести надо. Такие параметры указываются в функции как опциональные и по умолчанию указываются `None`. Также необходимо добавить логику на то, что если этот параметр `None`, то проводим тест фактически (и считаем значение этого параметра).
6. **Пишем докстринги**. Комментарии являются важной частью при написании любого кода. [Требования к комментариям (Раздел "Документация")](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=12039618894).
7. **Пишем юнит-тесты**. Иерархия юнит-тестов повторяет иерархию библиотеки. Файл с тестами к функции должен иметь префикс test_. Должно быть минимально необходимое для полного покрытия логики теста число юнит-тестов. Пример теста см. в `unit_tests/val_test/model_quality/test_valtest_metric.py`.

Пример теста см. `llm_val/val_test/model_quality/valtest_metric.py`.

## Пример с рассуждениями

Попробуем написать тест, используя план выше. Допустим, что необходимо написать тест *"2.1 Качество прогноза модели по ключевой метрике качества"*.

### 0. Определяемся с названием теста

Название теста можно сократить до `test_metric`, так как все, что мы делаем в логике теста - это считаем метрику. Но название должно начинаться с префикса `valtest_`, поэтому получаем `valtest_metric`. Из названия понятно, что тест нужен для подсчета метрики.

### 1. Создаем файл valtest_*название_теста*.py

С названием теста определились. Теперь необходимо создать файл с тестом. Тест 2.1 - это тест на качество модели(качество прогноза), поэтому в директории `llm_val/val_test/model_quality/` создаем файл `valtest_metric.py`.

### 2. Пишем логику теста

Первым делом определяем функцию и определяемся с параметрами функции. По параметрам нам нужен `sampler`(хранит в себе данные), `scorer`(считает метрику), `data_type`(тип данных, на которых производится расчет метрики. Hint: название атрибута в sampler'е), `metric_name`(название метрики), `semaphore_threshold`(границы светофора), `greater_is_better`(True, если чем больше метрика, тем лучше), `is_info`(True, если тест информативный). Также не забываем про `**kwargs` и типизацию. Получаем следующую сигнатуру функции:
```python
def valtest_metric(sampler: Sampler,
                   scorer: Scorer,
                   data_type: str='test',
                   metric_name: str='accuracy',
                   semaphore_threshold: tp.Tuple[float, float]=(0.2, 0.4),
                   greater_is_better: bool=True,
                   is_info: bool=False,
                   **kwargs) -> tp.Dict[str, tp.Dict[str, tp.Any]]:
```
Тест заключается в том, что необходимо посчитать метрику. Для этих целей у нас есть scorer и метод calc.
```python
metric_value = scorer.calc(sampler=sampler,
                           data_type=data_type,
                           metric_name=metric_name)[metric_name]
```
Вся логика теста готова🤪

По итогу получаем следующий код:
```python
def valtest_metric(sampler: Sampler,
                   scorer: Scorer,
                   data_type: str='test',
                   metric_name: str='accuracy',
                   semaphore_threshold: tp.Tuple[float, float]=(0.2, 0.4),
                   greater_is_better: bool=True,
                   is_info: bool=False,
                   **kwargs) -> tp.Dict[str, tp.Dict[str, tp.Any]]:
    metric_value = scorer.calc(sampler=sampler,
                               data_type=data_type,
                               metric_name=metric_name)[metric_name]
```

### 3. Создаем report

После того, как логика теста была написана, необходимо определиться, какие результаты мы хотим вынести в отчет. Результаты в отчет тянутся из возвращаемого словаря по ключу `report`. В отчет можно вынести следующие вещи: таблички, светофор и графики. Для создания report'а создаем отдельную функцию, в которой:
1. Определяется светофор;
2. Создаются "презентабельные" таблички;
3. Определяется, какие графики пойдут в отчет.

```python
def report_valtest_metric(metric_value: float,
                          data_type: str='test',
                          metric_name: str='accuracy',
                          semaphore_threshold: tp.Tuple[float, float]=(0.2, 0.4),
                          greater_is_better: bool=True,
                          is_info: bool=False):
    color = 'gray' if is_info else semaphore_by_threshold(value=metric_value, 
                                                          threshold=semaphore_threshold, 
                                                          greater_is_better=greater_is_better)
    res_table = pd.DataFrame({'Наименование метрики': [metric_name], 
                              f'Значение метрики на {data_type}': [metric_value],
                              'Результат теста': [color]})
    res_dict = {}
    res_dict['semaphore'] = color
    res_dict['result_plots'] = []
    res_dict['result_dataframes'] = [res_table]
    return res_dict
```
Вот и вся логика создания репорта😎

### 4. Сохраняем необходимые значения в precomputed

`precomputed` - это удобная фича для сохранения промежуточных результатов посчитанных внутри тестов. Для чего она нужна:
1. Пересчет теста без пересчета определенных значений;
2. Переиспользование уже подсчитанных значений в других тестах(сомнительно, но окей).
Попробуем в нашем тесте сохранить `metric_value`:

```python
precomputed = {'metric_value': metric_value}
```
И полученные `report` и `precomputed` вернуть из теста. На данной итерации получаем следующий код:
```python
def valtest_metric(sampler: Sampler,
                   scorer: Scorer,
                   data_type: str='test',
                   metric_name: str='accuracy',
                   semaphore_threshold: tp.Tuple[float, float]=(0.2, 0.4),
                   greater_is_better: bool=True,
                   is_info: bool=False,
                   **kwargs) -> tp.Dict[str, tp.Dict[str, tp.Any]]:
    metric_value = scorer.calc(sampler=sampler,
                               data_type=data_type,
                               metric_name=metric_name)[metric_name]
    report = report_valtest_metric(metric_value=metric_value,
                                   data_type=data_type,
                                   metric_name=metric_name,
                                   semaphore_threshold=semaphore_threshold,
                                   greater_is_better=greater_is_better,
                                   is_info=is_info)
    precomputed = {'metric_value': metric_value}
    return {'report': report, 
            'precomputed': precomputed}
```

### 5. Добавляем значения из precomputed в параметры функции

После того, как мы добавили сохранение значений в `precomputed`, необходимо(**с таким же названием, как и названия ключей в precomputed**) добавить параметры в функцию, чтобы можно было передать предподсчитанные значения. Также необходимо добавить проверку на перерасчет значения. Получаем следующий код:
```python
def valtest_metric(sampler: Sampler,
                   scorer: Scorer,
                   data_type: str='test',
                   metric_name: str='accuracy',
                   semaphore_threshold: tp.Tuple[float, float]=(0.2, 0.4),
                   greater_is_better: bool=True,
                   is_info: bool=False,
                   metric_value: tp.Optional[float]=None,
                   **kwargs) -> tp.Dict[str, tp.Dict[str, tp.Any]]:
    if metric_value is None:
        metric_value = scorer.calc(sampler=sampler,
                                   data_type=data_type,
                                   metric_name=metric_name)[metric_name]
    report = report_valtest_metric(metric_value=metric_value,
                                   data_type=data_type,
                                   metric_name=metric_name,
                                   semaphore_threshold=semaphore_threshold,
                                   greater_is_better=greater_is_better,
                                   is_info=is_info)
    precomputed = {'metric_value': metric_value}
    return {'report': report, 
            'precomputed': precomputed}
```

### 6. Пишем докстринги

Самая важная часть написания тестов - это написание документации. Подробнее посмотреть требования к документации можно [здесь (Раздел "Документация")](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=12039618894).
Код с комментариями выглядит следующим образом:
```python
def valtest_metric(sampler: Sampler,
                   scorer: Scorer,
                   data_type: str='test',
                   metric_name: str='accuracy',
                   semaphore_threshold: tp.Tuple[float, float]=(0.2, 0.4),
                   greater_is_better: bool=True,
                   is_info: bool=False,
                   metric_value: tp.Optional[float]=None,
                   **kwargs) -> tp.Dict[str, tp.Dict[str, tp.Any]]:
    '''
    Тест подсчета метрик
    Parameters:
        sampler: Объект класса Sampler
        scorer: Объект класса Scorer
        data_type: Тип данных, на которых производится расчет метрики. Hint: название атрибута в sampler'е
        metric_name: Название возвращаемой метрики
        semaphore_threshold: Границы светофора. Указываются в порядке возрастания
        greater_is_better: True, если чем больше метрика, тем лучше.
        is_info: True, если тест информативный
        metric_value: Опциональное значение метрики. 
            Если не None, то подсчет метрики не производится, и на основе переданного аргумента выставляется светофор.
    Return:
        Словарь в соответствии с концепцией llm_val
    '''
    if metric_value is None:
        metric_value = scorer.calc(sampler=sampler,
                                   data_type=data_type,
                                   metric_name=metric_name)[metric_name]
    report = report_valtest_metric(metric_value=metric_value,
                                   data_type=data_type,
                                   metric_name=metric_name,
                                   semaphore_threshold=semaphore_threshold,
                                   greater_is_better=greater_is_better,
                                   is_info=is_info)
    precomputed = {'metric_value': metric_value}
    return {'report': report, 
            'precomputed': precomputed}
```

### 7. Пишем юнит-тесты

Вторая самая важная часть в написании тестов - это написание юнит-тестов. Все файлы с юнит-тестами получают префикс test_. Например, если тест называется `valtest_metric`, то файл с юнит-тестами будет называться `test_valtest_metric`.

Обязательные(минимальные) проверки в юнит тестах:
1. Проверка на корректность возвращаемого словаря: есть `report` и `precomputed`;
2. Проверка на корректность report'а;
3. Проверка на корректность precomputed'а.

Эти проверки реализованы в файле `unit_tests/val_test/test_val_test.py`. Все, что нужно от пользователя - это добавить в словарь `TEST_PARAMS` указатель на функцию теста.

Юнит-тесты должны (по возможности) полностью покрывать логику теста. Поэтому, помимо минимальных проверок, необходимо реализовать различные проверки логики теста.

Пример юнит-теста можно посмотреть в `unit_tests/val_test/model_quality/test_valtest_stat_significance.py`